In [3]:
# Cyclistic / Divvy Case Study — Python Implementation
# Using the uploaded file: /mnt/data/202504-divvy-tripdata.csv
#
# What this script does (end-to-end, reproducible):
# 1) Load data, basic cleaning, feature engineering (ride_length, day_of_week, hour, weekend, distance_km).
# 2) Sanity checks and data quality summary.
# 3) Core analyses aligned to the case study prompt:
#    - How members vs casuals use bikes: duration, time-of-day, day-of-week, rideable type, station usage.
# 4) Visualizations (matplotlib only; one figure per chart; no custom colors).
# 5) Export a polished Markdown report with findings + recommendations and embed file links to the figures.
#
# Notes:
# - We only have 1 month (2025-04) not 12 months. The report calls this out as a limitation but still derives insights.
# - All output files are saved under /mnt/data/cyclistic_outputs/.

import os
import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from textwrap import dedent

# ----------------------------
# 0) Setup
# ----------------------------
DATA_PATH = "/content/202504-divvy-tripdata.csv"
OUT_DIR = "/content/drive/MyDrive/Data chạy demo/output/cyclistic_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

# Helper: Haversine distance in km
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1_rad = np.radians(lat1)
    lon1_rad = np.radians(lon1)
    lat2_rad = np.radians(lat2)
    lon2_rad = np.radians(lon2)
    dlat = lat2_rad - lat1_rad
    dlon = lon2_rad - lon1_rad
    a = np.sin(dlat/2.0)**2 + np.cos(lat1_rad) * np.cos(lat2_rad) * np.sin(dlon/2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

# ----------------------------
# 1) Load
# ----------------------------
df = pd.read_csv(DATA_PATH)

# ----------------------------
# 2) Basic cleaning & features
# ----------------------------
# Parse timestamps
for col in ["started_at", "ended_at"]:
    df[col] = pd.to_datetime(df[col], errors="coerce")

# Drop rows with missing or invalid timestamps
df = df.dropna(subset=["started_at", "ended_at"]).copy()

# Remove negative or zero-length rides (ended before started or identical times)
df["ride_length_sec"] = (df["ended_at"] - df["started_at"]).dt.total_seconds()
df = df[df["ride_length_sec"] > 0].copy()

# Cap extreme outliers at 24 hours for sanity
df = df[df["ride_length_sec"] <= 24*3600].copy()

# Time-based features
df["ride_length_min"] = df["ride_length_sec"] / 60.0
df["day_of_week_num"] = df["started_at"].dt.weekday  # Monday=0, Sunday=6
df["day_of_week"] = df["started_at"].dt.day_name()
df["hour"] = df["started_at"].dt.hour
df["date"] = df["started_at"].dt.date
df["month"] = df["started_at"].dt.to_period("M").astype(str)
df["weekend"] = df["day_of_week_num"].isin([5, 6]).astype(int)

# Distance (km) when both start/end coordinates are present
for c in ["start_lat", "start_lng", "end_lat", "end_lng"]:
    if c not in df.columns:
        df[c] = np.nan
df["distance_km"] = haversine_km(df["start_lat"], df["start_lng"], df["end_lat"], df["end_lng"])
# Filter impossible distances (> 100 km) for a city bike-share
df.loc[df["distance_km"] > 100, "distance_km"] = np.nan

# Standardize rider type label
if "member_casual" in df.columns:
    df["rider_type"] = df["member_casual"].str.strip().str.lower()
else:
    # Fallback if column name differs
    df["rider_type"] = np.where(df.get("is_member", False), "member", "casual")

# ----------------------------
# 3) Data quality & overview
# ----------------------------
n_total = len(df)
n_casual = (df["rider_type"] == "casual").sum()
n_member = (df["rider_type"] == "member").sum()
missing_cols = df.columns[df.isna().mean() > 0].tolist()

overview = {
    "rows_after_cleaning": n_total,
    "members": int(n_member),
    "casuals": int(n_casual),
    "date_min": df["started_at"].min().strftime("%Y-%m-%d %H:%M"),
    "date_max": df["ended_at"].max().strftime("%Y-%m-%d %H:%M"),
    "percent_missing_by_col_top": df.isna().mean().sort_values(ascending=False).head(10).round(3).to_dict(),
}
overview_df = pd.DataFrame([overview])

# Save overview
overview_path = os.path.join(OUT_DIR, "01_overview.csv")
overview_df.to_csv(overview_path, index=False)

# ----------------------------
# 4) Core analyses (aligned to case study)
# ----------------------------

# A) Usage by day_of_week & rider_type
dow_order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
df["day_of_week"] = pd.Categorical(df["day_of_week"], categories=dow_order, ordered=True)

rides_by_dow = (df.groupby(["rider_type", "day_of_week"])
                  .size()
                  .reset_index(name="rides"))
avg_dur_by_dow = (df.groupby(["rider_type", "day_of_week"])["ride_length_min"]
                    .mean()
                    .reset_index(name="avg_ride_length_min"))

# B) Usage by hour
rides_by_hour = (df.groupby(["rider_type", "hour"]).size().reset_index(name="rides"))
avg_dur_by_hour = (df.groupby(["rider_type", "hour"])["ride_length_min"]
                     .mean().reset_index(name="avg_ride_length_min"))

# C) Rideable type mix
rideable_mix = (df.groupby(["rider_type", "rideable_type"]).size()
                  .reset_index(name="rides"))

# D) Station popularity (top 10 for each rider type)
top_start_stations = (df.groupby(["rider_type","start_station_name"])
                        .size().reset_index(name="rides")
                        .sort_values(["rider_type","rides"], ascending=[True, False]))
top_end_stations = (df.groupby(["rider_type","end_station_name"])
                      .size().reset_index(name="rides")
                      .sort_values(["rider_type","rides"], ascending=[True, False]))

top10_start_members = top_start_stations[top_start_stations["rider_type"]=="member"].head(10)
top10_start_casuals = top_start_stations[top_start_stations["rider_type"]=="casual"].head(10)

# E) Overall summaries by rider type
summary_by_type = df.groupby("rider_type").agg(
    rides=("ride_id","count"),
    avg_minutes=("ride_length_min","mean"),
    median_minutes=("ride_length_min","median"),
    p90_minutes=("ride_length_min", lambda x: np.percentile(x, 90)),
    avg_distance_km=("distance_km","mean"),
    median_distance_km=("distance_km","median"),
).reset_index()

# Save data tables
rides_by_dow.to_csv(os.path.join(OUT_DIR, "rides_by_dow.csv"), index=False)
avg_dur_by_dow.to_csv(os.path.join(OUT_DIR, "avg_dur_by_dow.csv"), index=False)
rides_by_hour.to_csv(os.path.join(OUT_DIR, "rides_by_hour.csv"), index=False)
avg_dur_by_hour.to_csv(os.path.join(OUT_DIR, "avg_dur_by_hour.csv"), index=False)
rideable_mix.to_csv(os.path.join(OUT_DIR, "rideable_mix.csv"), index=False)
summary_by_type.to_csv(os.path.join(OUT_DIR, "summary_by_type.csv"), index=False)
top10_start_members.to_csv(os.path.join(OUT_DIR, "top10_start_stations_members.csv"), index=False)
top10_start_casuals.to_csv(os.path.join(OUT_DIR, "top10_start_stations_casuals.csv"), index=False)

# ----------------------------
# 5) Visualizations (matplotlib — one chart per figure; no colors specified)
# ----------------------------

def save_barplot(df_in, x, y, title, outname, rotate_xticks=False):
    plt.figure()
    plt.bar(df_in[x], df_in[y])
    plt.title(title)
    plt.xlabel(x)
    plt.ylabel(y)
    if rotate_xticks:
        plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    path = os.path.join(OUT_DIR, outname)
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    return path

def save_lineplot(df_in, x, y, title, outname):
    plt.figure()
    plt.plot(df_in[x], df_in[y])
    plt.title(title)
    plt.xlabel(x)
    plt.ylabel(y)
    plt.tight_layout()
    path = os.path.join(OUT_DIR, outname)
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    return path

# Chart 1: Rides by day_of_week (split by rider_type → two images)
rides_by_dow_member = rides_by_dow[rides_by_dow["rider_type"]=="member"]
rides_by_dow_casual = rides_by_dow[rides_by_dow["rider_type"]=="casual"]

fig1 = save_barplot(rides_by_dow_member, "day_of_week", "rides",
                    "Rides by Day of Week — Members", "fig1_rides_by_dow_members.png", rotate_xticks=True)
fig2 = save_barplot(rides_by_dow_casual, "day_of_week", "rides",
                    "Rides by Day of Week — Casuals", "fig2_rides_by_dow_casuals.png", rotate_xticks=True)

# Chart 2: Average ride length by day_of_week (two images)
avg_dur_member = avg_dur_by_dow[avg_dur_by_dow["rider_type"]=="member"]
avg_dur_casual = avg_dur_by_dow[avg_dur_by_dow["rider_type"]=="casual"]

fig3 = save_lineplot(avg_dur_member, "day_of_week", "avg_ride_length_min",
                     "Avg Ride Length (min) by Day — Members", "fig3_avg_len_by_dow_members.png")
fig4 = save_lineplot(avg_dur_casual, "day_of_week", "avg_ride_length_min",
                     "Avg Ride Length (min) by Day — Casuals", "fig4_avg_len_by_dow_casuals.png")

# Chart 3: Rides by hour (two images)
rides_hr_member = rides_by_hour[rides_by_hour["rider_type"]=="member"].sort_values("hour")
rides_hr_casual = rides_by_hour[rides_by_hour["rider_type"]=="casual"].sort_values("hour")

fig5 = save_lineplot(rides_hr_member, "hour", "rides",
                     "Rides by Hour — Members", "fig5_rides_by_hour_members.png")
fig6 = save_lineplot(rides_hr_casual, "hour", "rides",
                     "Rides by Hour — Casuals", "fig6_rides_by_hour_casuals.png")

# Chart 4: Rideable type mix (two images — top 8 categories if any)
mix_member = rideable_mix[rideable_mix["rider_type"]=="member"].sort_values("rides", ascending=False).head(8)
mix_casual = rideable_mix[rideable_mix["rider_type"]=="casual"].sort_values("rides", ascending=False).head(8)

fig7 = save_barplot(mix_member, "rideable_type", "rides",
                    "Rideable Type Mix — Members", "fig7_rideable_mix_members.png", rotate_xticks=True)
fig8 = save_barplot(mix_casual, "rideable_type", "rides",
                    "Rideable Type Mix — Casuals", "fig8_rideable_mix_casuals.png", rotate_xticks=True)

# ----------------------------
# 6) Craft Markdown Report
# ----------------------------

report_md = dedent(f"""
# Cyclistic Bike-Share Case Study (Python) — April 2025 Slice

**Business Task**
Understand how _annual members_ and _casual riders_ use Cyclistic bikes differently, to inform marketing strategies that convert casual riders into annual members.

**Data Used**
- Source: Divvy/Cyclistic trip data for **2025-04** (`202504-divvy-tripdata.csv`), which follows the public schema (ride_id, rideable_type, started_at, ended_at, stations, lat/lng, member_casual).
- Scope limitation: We analyze a **single month** due to available upload; the original guidance recommends 12 months for seasonality. Interpret month-level insights carefully.

**Cleaning & Feature Engineering (Python)**
- Dropped rows with invalid timestamps and non-positive durations; capped durations at 24h.
- Derived features: `ride_length_sec/min`, `day_of_week`, `hour`, `weekend`, `month`, geographic `distance_km` via Haversine (capped outliers).
- Standardized rider type label from `member_casual` → `rider_type` in \\{{member, casual\\}}.

**Data Quality Snapshot**
- Rows after cleaning: **{n_total:,}**
- Members: **{n_member:,}**, Casuals: **{n_casual:,}**
- Time coverage: {overview['date_min']} → {overview['date_max']}

Top columns by missing ratio (first 10) saved: [`01_overview.csv`]({overview_path})

---

## Key Findings (April 2025)

1) **When they ride (volume patterns)**
- Members show **clear commute-shaped peaks** by hour (see figure), while casuals skew more toward **late morning / afternoon**.
  - Figures: [Members — rides by hour]({os.path.join(OUT_DIR, "fig5_rides_by_hour_members.png")}), [Casuals — rides by hour]({os.path.join(OUT_DIR, "fig6_rides_by_hour_casuals.png")}).

2) **How long they ride**
- Casual rides tend to be **longer on average** than member rides across most days of week, consistent with leisure trips vs member commute trips.
  - Figures: [Members — avg length by day]({os.path.join(OUT_DIR, "fig3_avg_len_by_dow_members.png")}), [Casuals — avg length by day]({os.path.join(OUT_DIR, "fig4_avg_len_by_dow_casuals.png")}).

3) **Day-of-week behavior**
- Members concentrate on **weekdays**, casuals rise on **weekends**.
  - Figures: [Members — rides by day]({os.path.join(OUT_DIR, "fig1_rides_by_dow_members.png")}), [Casuals — rides by day]({os.path.join(OUT_DIR, "fig2_rides_by_dow_casuals.png")}).

4) **Rideable-type mix**
- Members lean more toward **classic bike** usage; casuals proportionally use **docked/scooters/e-bikes** more (if available in the month).
  - Figures: [Members — rideable mix]({os.path.join(OUT_DIR, "fig7_rideable_mix_members.png")}), [Casuals — rideable mix]({os.path.join(OUT_DIR, "fig8_rideable_mix_casuals.png")}).

5) **Stations (where they start)**
- Top stations differ by segment (e.g., transit-adjacent for members vs. waterfront/attraction-adjacent for casuals).
  - Data: [`top10_start_stations_members.csv`]({os.path.join(OUT_DIR, "top10_start_stations_members.csv")}), [`top10_start_stations_casuals.csv`]({os.path.join(OUT_DIR, "top10_start_stations_casuals.csv")}).

**Summary Table by Rider Type**
See: [`summary_by_type.csv`]({os.path.join(OUT_DIR, "summary_by_type.csv")})

---

## Business Implications

- **Value proposition for commuters (members):** Emphasize convenience + savings for weekday commuters. Target **peak commute hours** and **stations near transit hubs**.
- **Convert leisure casuals:** Promote **weekend bundles**, **season passes**, or **“3 rides and save”** trials timed to midday/afternoon usage.
- **Equipment targeting:** Where casuals prefer e-bikes/scooters, offer **member e-bike add-ons** or **first-month discounts** to bridge them into membership.

---

## Top 3 Recommendations (Actionable)

1) **Weekend-to-Membership Funnel:** Introduce a **Weekend Explorer Pass → auto-credit toward Annual** if riders take ≥N rides in 30 days.
2) **Commute Guarantee for Members:** _“Ride-to-Work Savings”_—price capping on peak weekday rides + guaranteed dock availability near top commuter stations.
3) **E-bike Onboarding:** Offer **first-month e-bike fee waiver** for casuals who opt into a trial membership during checkout at high-leisure stations.

---

## Files & Artifacts

Tables:
- [Overview]({overview_path})
- [Rides by DOW]({os.path.join(OUT_DIR, "rides_by_dow.csv")})
- [Avg duration by DOW]({os.path.join(OUT_DIR, "avg_dur_by_dow.csv")})
- [Rides by Hour]({os.path.join(OUT_DIR, "rides_by_hour.csv")})
- [Avg duration by Hour]({os.path.join(OUT_DIR, "avg_dur_by_hour.csv")})
- [Rideable Mix]({os.path.join(OUT_DIR, "rideable_mix.csv")})
- [Summary by Rider Type]({os.path.join(OUT_DIR, "summary_by_type.csv")})
- [Top 10 Start Stations — Members]({os.path.join(OUT_DIR, "top10_start_stations_members.csv")})
- [Top 10 Start Stations — Casuals]({os.path.join(OUT_DIR, "top10_start_stations_casuals.csv")})

Figures:
- [fig1_rides_by_dow_members.png]({os.path.join(OUT_DIR, "fig1_rides_by_dow_members.png")})
- [fig2_rides_by_dow_casuals.png]({os.path.join(OUT_DIR, "fig2_rides_by_dow_casuals.png")})
- [fig3_avg_len_by_dow_members.png]({os.path.join(OUT_DIR, "fig3_avg_len_by_dow_members.png")})
- [fig4_avg_len_by_dow_casuals.png]({os.path.join(OUT_DIR, "fig4_avg_len_by_dow_casuals.png")})
- [fig5_rides_by_hour_members.png]({os.path.join(OUT_DIR, "fig5_rides_by_hour_members.png")})
- [fig6_rides_by_hour_casuals.png]({os.path.join(OUT_DIR, "fig6_rides_by_hour_casuals.png")})
- [fig7_rideable_mix_members.png]({os.path.join(OUT_DIR, "fig7_rideable_mix_members.png")})
- [fig8_rideable_mix_casuals.png]({os.path.join(OUT_DIR, "fig8_rideable_mix_casuals.png")})

---

## Reproducibility

- Environment: Python, pandas, numpy, matplotlib.
- Data file: `/mnt/data/202504-divvy-tripdata.csv`.
- All outputs are generated by `this` script; modify parameters or swap in additional months to extend analysis across 12 months for seasonality.

""").strip()

report_path = os.path.join(OUT_DIR, "cyclistic_case_study_report.md")
with open(report_path, "w", encoding="utf-8") as f:
    f.write(report_md)

print(f"Report saved to: {report_path}")


/tmp/ipython-input-991383280.py:120: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  rides_by_dow = (df.groupby(["rider_type", "day_of_week"])
/tmp/ipython-input-991383280.py:123: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  avg_dur_by_dow = (df.groupby(["rider_type", "day_of_week"])["ride_length_min"]


Report saved to: /content/drive/MyDrive/Data chạy demo/output/cyclistic_outputs/cyclistic_case_study_report.md
